# Computational Notebook 14: Consensus Simulations

## Overview

Consensus mechanisms are the algorithms that allow distributed networks to agree on a single version of truth without a central authority. They are the beating heart of every blockchain, determining its security, performance, and decentralization properties. This notebook builds working simulations of major consensus mechanisms -- Proof of Work (PoW), Proof of Stake (PoS), and Practical Byzantine Fault Tolerance (PBFT) -- from first principles. We explore the Byzantine Generals Problem, simulate mining and validator selection, analyze finality properties, model attack scenarios, and compare mechanisms across key performance dimensions.

## Prerequisites
- **Notebook 01**: Cryptographic Primitives (hash functions, digital signatures)
- **Notebook 07**: Mining Economics (hashrate, difficulty, block rewards)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Understand the Byzantine Generals Problem and the FLP impossibility result
2. Simulate Proof of Work mining with adjustable difficulty and measure block time distributions
3. Implement Proof of Stake validator selection weighted by stake and simulate slashing
4. Build a PBFT simulator with three-phase commit and analyze message complexity
5. Compare probabilistic finality (PoW) with deterministic finality (BFT)
6. Model 51% attack scenarios and calculate attack success probabilities
7. Benchmark consensus mechanisms across throughput, latency, finality, and decentralization

**Estimated Time:** 4-6 hours

**Related Content:** [Section 02: Bitcoin Deep Dive](../sections/02-bitcoin-deep-dive.md) | [Section 05: Platform Comparison](../sections/05-platform-comparison.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import time
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook simulates blockchain consensus mechanisms.")

---
## 1. Byzantine Generals Problem

The Byzantine Generals Problem (BGP), formulated by Lamport, Shostak, and Pease in 1982, asks: how can a group of distributed actors agree on a plan of action when some actors may be traitors?

> **Definition: Byzantine Fault Tolerance (BFT)** -- The ability of a distributed system to reach consensus even when some nodes behave maliciously (sending conflicting information, failing to respond, or actively trying to disrupt consensus).

Key results:
- With **oral messages** (no authentication): consensus requires $n \geq 3f + 1$ nodes, where $f$ is the number of faulty nodes
- With **signed messages** (authenticated): consensus is possible with any $n \geq f + 1$

> **Definition: FLP Impossibility** -- The Fischer-Lynch-Paterson result (1985) proves that in an asynchronous distributed system, it is impossible to guarantee consensus if even one node can fail. Practical systems work around this by using partial synchrony assumptions.

**Source:** Lamport, L., Shostak, R., & Pease, M. (1982). "The Byzantine Generals Problem." *ACM TOPLAS*.

In [ ]:
def byzantine_generals_simulation(n_generals: int, n_traitors: int,
                                   rounds: int = 100) -> Dict[str, float]:
    """Simulate the Byzantine Generals Problem.
    
    Loyal generals try to agree on 'attack' or 'retreat'.
    Traitors send conflicting messages to different generals.
    
    Args:
        n_generals: Total number of generals
        n_traitors: Number of traitor generals
        rounds: Number of simulation rounds
    
    Returns:
        Consensus success rate and other metrics
    """
    n_loyal = n_generals - n_traitors
    successes = 0
    
    for _ in range(rounds):
        # Commander (general 0) sends the true order
        true_order = np.random.choice(['attack', 'retreat'])
        
        # Each loyal general collects messages
        loyal_decisions = []
        
        for i in range(n_loyal):
            received = []
            # Messages from loyal generals (truthful)
            for j in range(n_loyal):
                received.append(true_order)
            # Messages from traitors (random/conflicting)
            for j in range(n_traitors):
                received.append(np.random.choice(['attack', 'retreat']))
            
            # Majority vote
            attacks = sum(1 for m in received if m == 'attack')
            decision = 'attack' if attacks > len(received) / 2 else 'retreat'
            loyal_decisions.append(decision)
        
        # Check if all loyal generals agreed
        if len(set(loyal_decisions)) == 1:
            successes += 1
    
    return {
        'n_generals': n_generals,
        'n_traitors': n_traitors,
        'n_loyal': n_loyal,
        'fault_tolerance': n_traitors / n_generals,
        'consensus_rate': successes / rounds,
        'meets_bft_threshold': n_generals >= 3 * n_traitors + 1
    }


print("=" * 60)
print("BYZANTINE GENERALS PROBLEM")
print("=" * 60)

np.random.seed(42)

print(f"\n{'Generals':>10} {'Traitors':>10} {'Loyal':>7} {'BFT OK?':>9} {'Consensus':>11}")
print("-" * 50)

test_cases = [
    (4, 1), (4, 2), (7, 2), (7, 3),
    (10, 3), (10, 4), (13, 4), (20, 6),
]

for n, f in test_cases:
    result = byzantine_generals_simulation(n, f, rounds=1000)
    bft_ok = 'Yes' if result['meets_bft_threshold'] else 'No'
    print(f"{n:>10} {f:>10} {n-f:>7} {bft_ok:>9} {result['consensus_rate']:>10.1%}")

print(f"\nBFT threshold: n >= 3f + 1 (can tolerate up to f = (n-1)/3 faults)")

---
## 2. Proof of Work (PoW) Simulation

Proof of Work requires miners to find a nonce such that the hash of the block header falls below a target value. The probability of finding a valid block is proportional to computational power.

$$\text{target} = \frac{2^{256}}{\text{difficulty}}$$

$$P(\text{valid hash}) = \frac{\text{target}}{2^{256}} = \frac{1}{\text{difficulty}}$$

Block times follow a geometric distribution (memoryless property), which approximates an exponential distribution for large difficulty:

$$P(T > t) = e^{-\lambda t}$$

where $\lambda = \text{hashrate} / \text{difficulty}$.

**Source:** Nakamoto, S. (2008). "Bitcoin: A Peer-to-Peer Electronic Cash System."

In [ ]:
class ProofOfWorkSimulator:
    """Simplified Proof of Work mining simulator."""
    
    def __init__(self, difficulty: int = 4, target_block_time: float = 10.0) -> None:
        """Initialize PoW simulator.
        
        Args:
            difficulty: Number of leading zero hex digits required
            target_block_time: Target seconds between blocks
        """
        self.difficulty = difficulty
        self.target = '0' * difficulty
        self.target_block_time = target_block_time
        self.chain: List[Dict] = []
        self.block_times: List[float] = []
    
    def mine_block(self, data: str, prev_hash: str = "0" * 64) -> Dict:
        """Mine a single block by finding a valid nonce."""
        nonce = 0
        start = time.time()
        
        while True:
            block_str = f"{prev_hash}{data}{nonce}"
            block_hash = hashlib.sha256(block_str.encode()).hexdigest()
            
            if block_hash[:self.difficulty] == self.target:
                elapsed = time.time() - start
                block = {
                    'data': data,
                    'nonce': nonce,
                    'hash': block_hash,
                    'prev_hash': prev_hash,
                    'time': elapsed,
                    'attempts': nonce + 1
                }
                self.chain.append(block)
                self.block_times.append(elapsed)
                return block
            
            nonce += 1
    
    def mine_blocks(self, n: int) -> List[Dict]:
        """Mine n blocks sequentially."""
        blocks = []
        prev_hash = "0" * 64
        for i in range(n):
            block = self.mine_block(f"Block {i+1}", prev_hash)
            prev_hash = block['hash']
            blocks.append(block)
        return blocks


# Mine several blocks with low difficulty for demonstration
print("=" * 60)
print("PROOF OF WORK MINING SIMULATION")
print("=" * 60)

pow_sim = ProofOfWorkSimulator(difficulty=4)  # 4 hex zeros
print(f"Difficulty: {pow_sim.difficulty} leading zeros")
print(f"Target: hash must start with '{pow_sim.target}'")
print(f"Expected attempts: ~{16**pow_sim.difficulty:,}\n")

blocks = pow_sim.mine_blocks(10)

print(f"{'Block':>6} {'Nonce':>10} {'Attempts':>10} {'Time (s)':>10} {'Hash (first 20)':>22}")
print("-" * 62)
for i, b in enumerate(blocks):
    print(f"{i+1:>6} {b['nonce']:>10,} {b['attempts']:>10,} {b['time']:>9.3f}s {b['hash'][:20]}...")

print(f"\nAverage attempts: {np.mean([b['attempts'] for b in blocks]):,.0f}")
print(f"Average block time: {np.mean(pow_sim.block_times):.3f}s")

In [ ]:
# Analyze block time distribution
# Use statistical simulation instead of actual mining for speed
np.random.seed(42)

# Bitcoin-like parameters
target_time = 600  # 10 minutes
n_blocks = 10000

# Block times follow exponential distribution
simulated_times = np.random.exponential(target_time, n_blocks)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Block time distribution
axes[0, 0].hist(simulated_times / 60, bins=80, density=True, alpha=0.7, color='blue')
x = np.linspace(0, 60, 200)
theoretical = (1/target_time) * np.exp(-x * 60 / target_time)
axes[0, 0].plot(x, theoretical * 60, 'r-', linewidth=2, label='Exponential PDF')
axes[0, 0].axvline(x=target_time/60, color='green', linestyle='--',
                    label=f'Target: {target_time/60:.0f} min')
axes[0, 0].set_xlabel('Block Time (minutes)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Block Time Distribution (Exponential)')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 60)

# CDF - probability of finding block within time
sorted_times = np.sort(simulated_times)
cdf = np.arange(1, len(sorted_times) + 1) / len(sorted_times)
axes[0, 1].plot(sorted_times / 60, cdf, 'b-', linewidth=2)
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
axes[0, 1].axvline(x=target_time * np.log(2) / 60, color='red',
                    linestyle='--', alpha=0.5, label=f'Median: {target_time*np.log(2)/60:.1f} min')
axes[0, 1].set_xlabel('Time (minutes)')
axes[0, 1].set_ylabel('P(block found)')
axes[0, 1].set_title('Cumulative Distribution')
axes[0, 1].legend()
axes[0, 1].set_xlim(0, 60)

# Selfish mining: withholding blocks
honest_shares = np.linspace(0, 1, 100)
selfish_shares = 1 - honest_shares

# Selfish mining revenue (simplified model)
# With gamma=0.5 (50% of network hears attacker's block first)
gamma = 0.5
selfish_revenue = []
honest_revenue = []
for alpha in selfish_shares:
    if alpha < 0.5:
        # Simplified selfish mining formula
        rev = (alpha * (1 - alpha)**2 * (4*alpha + gamma*(1-2*alpha)) - alpha**3) / \
              (1 - alpha*(1 + (2-alpha)*alpha))
        rev = max(0, min(alpha + rev, 1))  # Clamp
        selfish_revenue.append(rev)
    else:
        selfish_revenue.append(1.0)
    honest_revenue.append(alpha)  # Honest mining: proportional

axes[1, 0].plot(selfish_shares * 100, np.array(honest_revenue) * 100,
                'g-', linewidth=2, label='Honest mining')
axes[1, 0].plot(selfish_shares * 100, np.array(selfish_revenue) * 100,
                'r-', linewidth=2, label='Selfish mining')
axes[1, 0].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Proportional')
axes[1, 0].set_xlabel('Attacker Hashrate (%)')
axes[1, 0].set_ylabel('Revenue Share (%)')
axes[1, 0].set_title('Selfish Mining Profitability')
axes[1, 0].legend()

# Difficulty adjustment simulation
difficulty = 1.0
hashrate_changes = np.random.normal(1.02, 0.05, 100)  # Hashrate growth
difficulties = [difficulty]
actual_times = []

for change in hashrate_changes:
    # Actual block time affected by hashrate change
    actual_time = target_time / change
    actual_times.append(actual_time)
    # Adjust difficulty every 2016 blocks (simplified to every step)
    difficulty *= actual_time / target_time
    difficulty = max(0.1, difficulty)  # Floor
    difficulties.append(difficulty)

axes[1, 1].plot(difficulties, 'b-', linewidth=2)
axes[1, 1].set_xlabel('Adjustment Period')
axes[1, 1].set_ylabel('Relative Difficulty')
axes[1, 1].set_title('Difficulty Adjustment Over Time')

plt.tight_layout()
plt.savefig('/tmp/pow_simulation.png', dpi=100, bbox_inches='tight')
plt.show()
print("PoW simulation complete.")
print(f"Median block time ({target_time*np.log(2)/60:.1f} min) < Mean ({target_time/60:.0f} min) due to exponential distribution.")

---
## 3. Proof of Stake (PoS) Simulation

In Proof of Stake, validators are selected to propose blocks proportional to their staked tokens. There is no energy-intensive mining.

> **Definition: Proof of Stake (PoS)** -- A consensus mechanism where validators lock tokens as collateral ("stake") and are selected to propose and attest to blocks proportional to their stake. Misbehavior results in "slashing" (loss of staked tokens).

> **Definition: Slashing** -- The penalty mechanism in PoS systems where a validator loses part or all of their staked tokens for provable misbehavior (double-signing, prolonged downtime, etc.).

> **Definition: Nothing-at-Stake Problem** -- In naive PoS, validators have no cost to validating on multiple competing forks simultaneously, since there's no computational work. This is solved by slashing conditions that punish equivocation.

**Source:** Buterin, V. & Griffith, V. (2019). "Casper the Friendly Finality Gadget." *arXiv:1710.09437*.

In [ ]:
@dataclass
class Validator:
    """A PoS validator."""
    address: str
    stake: float
    blocks_proposed: int = 0
    attestations: int = 0
    slashings: int = 0
    total_rewards: float = 0.0
    total_penalties: float = 0.0
    active: bool = True


class ProofOfStakeSimulator:
    """Ethereum-style PoS simulator."""
    
    def __init__(self, validators: Dict[str, float],
                 block_reward: float = 0.05,
                 slash_penalty: float = 0.1,
                 min_stake: float = 32.0) -> None:
        """Initialize PoS simulator.
        
        Args:
            validators: {address: stake_amount}
            block_reward: Reward per block proposed (as fraction of stake)
            slash_penalty: Penalty for slashable offense (fraction of stake)
            min_stake: Minimum stake to be a validator (32 ETH on Ethereum)
        """
        self.validators = {}
        for addr, stake in validators.items():
            if stake >= min_stake:
                self.validators[addr] = Validator(addr, stake)
        self.block_reward_rate = block_reward
        self.slash_penalty_rate = slash_penalty
        self.min_stake = min_stake
        self.total_stake = sum(v.stake for v in self.validators.values())
        self.block_history: List[str] = []
    
    def select_proposer(self) -> str:
        """Select block proposer weighted by stake."""
        active = {a: v for a, v in self.validators.items() if v.active}
        addresses = list(active.keys())
        stakes = np.array([active[a].stake for a in addresses])
        probs = stakes / stakes.sum()
        return np.random.choice(addresses, p=probs)
    
    def propose_block(self) -> str:
        """Simulate block proposal."""
        proposer = self.select_proposer()
        v = self.validators[proposer]
        reward = v.stake * self.block_reward_rate / 365  # Daily rate
        v.blocks_proposed += 1
        v.total_rewards += reward
        v.stake += reward
        self.block_history.append(proposer)
        return proposer
    
    def slash_validator(self, address: str, reason: str = "double-sign") -> float:
        """Slash a validator for misbehavior."""
        v = self.validators[address]
        penalty = v.stake * self.slash_penalty_rate
        v.stake -= penalty
        v.total_penalties += penalty
        v.slashings += 1
        if v.stake < self.min_stake:
            v.active = False
        return penalty
    
    def simulate_epoch(self, blocks_per_epoch: int = 32) -> Dict:
        """Simulate one epoch of block production."""
        proposers = []
        for _ in range(blocks_per_epoch):
            proposers.append(self.propose_block())
        
        unique = len(set(proposers))
        return {'blocks': blocks_per_epoch, 'unique_proposers': unique}


# Create a validator set
np.random.seed(42)
n_validators = 100

# Most validators have 32 ETH, some whales have more
stakes = {}
for i in range(n_validators):
    if i < 5:  # 5 large stakers
        stakes[f"validator_{i:03d}"] = 32 * np.random.uniform(10, 100)
    else:
        stakes[f"validator_{i:03d}"] = 32 * np.random.uniform(1, 3)

pos_sim = ProofOfStakeSimulator(stakes)

print("=" * 60)
print("PROOF OF STAKE SIMULATION")
print("=" * 60)
print(f"Validators: {len(pos_sim.validators)}")
print(f"Total staked: {pos_sim.total_stake:,.0f} ETH")
print(f"Min stake: {pos_sim.min_stake} ETH")

# Simulate 365 epochs (1 year with ~1 epoch/day)
for _ in range(365):
    pos_sim.simulate_epoch(32)

# Results
validators_sorted = sorted(pos_sim.validators.values(),
                           key=lambda v: v.blocks_proposed, reverse=True)

print(f"\nAfter 1 year ({365 * 32:,} blocks):")
print(f"\n{'Validator':<16} {'Stake':>10} {'Blocks':>8} {'Rewards':>10} {'Share':>7}")
print("-" * 55)
total_blocks = sum(v.blocks_proposed for v in validators_sorted)
for v in validators_sorted[:10]:
    share = v.blocks_proposed / total_blocks * 100
    print(f"{v.address:<16} {v.stake:>8,.0f} ETH {v.blocks_proposed:>8,} "
          f"{v.total_rewards:>8,.1f} ETH {share:>5.1f}%")

In [ ]:
# Demonstrate nothing-at-stake and slashing
print("\n" + "=" * 60)
print("NOTHING-AT-STAKE & SLASHING DEMONSTRATION")
print("=" * 60)

# Simulate a fork where a validator signs both chains
cheater = validators_sorted[0].address
cheater_stake_before = pos_sim.validators[cheater].stake

print(f"\nValidator {cheater} detected double-signing on fork!")
print(f"  Stake before: {cheater_stake_before:,.1f} ETH")

penalty = pos_sim.slash_validator(cheater, "double-sign")
cheater_stake_after = pos_sim.validators[cheater].stake

print(f"  Penalty: {penalty:,.1f} ETH ({pos_sim.slash_penalty_rate:.0%} of stake)")
print(f"  Stake after: {cheater_stake_after:,.1f} ETH")
print(f"  Still active: {pos_sim.validators[cheater].active}")

# Visualize stake distribution and block proposal fairness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stake vs blocks proposed
stake_vals = [v.stake for v in validators_sorted]
block_vals = [v.blocks_proposed for v in validators_sorted]
axes[0].scatter(stake_vals, block_vals, alpha=0.5, s=30)
axes[0].set_xlabel('Stake (ETH)')
axes[0].set_ylabel('Blocks Proposed')
axes[0].set_title('PoS: Stake vs Block Production')

# Fit line
z = np.polyfit(stake_vals, block_vals, 1)
x_line = np.linspace(min(stake_vals), max(stake_vals), 100)
axes[0].plot(x_line, np.polyval(z, x_line), 'r--', label='Linear fit')
axes[0].legend()

# Cumulative stake distribution
sorted_stakes = np.sort([v.stake for v in pos_sim.validators.values()])[::-1]
cum_stake = np.cumsum(sorted_stakes) / sum(sorted_stakes) * 100
axes[1].plot(range(1, len(cum_stake) + 1), cum_stake, 'b-', linewidth=2)
axes[1].axhline(y=33.3, color='orange', linestyle='--', alpha=0.7, label='33% (liveness threshold)')
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50% (safety threshold)')
axes[1].axhline(y=66.7, color='darkred', linestyle='--', alpha=0.7, label='67% (finality threshold)')
axes[1].set_xlabel('Number of Validators')
axes[1].set_ylabel('Cumulative Stake (%)')
axes[1].set_title('Stake Concentration')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/pos_simulation.png', dpi=100, bbox_inches='tight')
plt.show()
print("PoS simulation complete. Block production is proportional to stake.")

---
## 4. PBFT (Practical Byzantine Fault Tolerance)

PBFT, proposed by Castro and Liskov (1999), achieves consensus in three phases:

1. **Pre-Prepare**: Leader broadcasts the proposed block to all validators
2. **Prepare**: Each validator broadcasts a PREPARE message to all others
3. **Commit**: After receiving 2f+1 PREPARE messages, validators broadcast COMMIT

Consensus is reached when a validator receives 2f+1 COMMIT messages.

> **Definition: PBFT Message Complexity** -- PBFT requires $O(n^2)$ messages per consensus round, where $n$ is the number of validators. This quadratic scaling limits PBFT to relatively small validator sets (typically <100).

Fault tolerance: $n \geq 3f + 1$, so a 4-node system tolerates 1 fault.

**Source:** Castro, M. & Liskov, B. (1999). "Practical Byzantine Fault Tolerance." *OSDI 1999*.

In [ ]:
class PBFTSimulator:
    """Simplified PBFT consensus simulator."""
    
    def __init__(self, n_nodes: int, n_faulty: int = 0) -> None:
        """Initialize PBFT simulator.
        
        Args:
            n_nodes: Total number of nodes
            n_faulty: Number of Byzantine (faulty) nodes
        """
        self.n = n_nodes
        self.f = n_faulty
        self.quorum = 2 * n_faulty + 1  # 2f + 1
        self.max_faults = (n_nodes - 1) // 3
        self.messages_sent = 0
        self.rounds_completed = 0
        self.consensus_history: List[Dict] = []
    
    def run_consensus_round(self, proposal: str) -> Dict:
        """Simulate one PBFT consensus round."""
        n_honest = self.n - self.f
        msgs = 0
        
        # Phase 1: Pre-Prepare (leader -> all)
        pre_prepare_msgs = self.n - 1  # Leader sends to all others
        msgs += pre_prepare_msgs
        
        # Phase 2: Prepare (each -> all)
        # Honest nodes send PREPARE to all others
        prepare_msgs = n_honest * (self.n - 1)
        msgs += prepare_msgs
        
        # Check if enough PREPAREs received (need 2f+1 matching)
        prepare_received = n_honest  # Honest nodes all agree
        prepare_ok = prepare_received >= self.quorum
        
        # Phase 3: Commit (each -> all)
        if prepare_ok:
            commit_msgs = n_honest * (self.n - 1)
            msgs += commit_msgs
            
            commit_received = n_honest
            commit_ok = commit_received >= self.quorum
        else:
            commit_ok = False
        
        self.messages_sent += msgs
        self.rounds_completed += 1
        
        result = {
            'round': self.rounds_completed,
            'proposal': proposal,
            'consensus': commit_ok,
            'messages': msgs,
            'honest_nodes': n_honest,
            'prepare_ok': prepare_ok,
            'commit_ok': commit_ok
        }
        self.consensus_history.append(result)
        return result


print("=" * 60)
print("PBFT CONSENSUS SIMULATION")
print("=" * 60)

# Test different configurations
print(f"\n{'Nodes':>7} {'Faulty':>8} {'Max f':>7} {'Safe?':>7} {'Consensus':>11} {'Messages':>10}")
print("-" * 55)

configs = [
    (4, 0), (4, 1), (4, 2),
    (7, 0), (7, 2), (7, 3),
    (10, 3), (10, 4),
    (25, 8), (100, 33),
]

for n, f in configs:
    pbft = PBFTSimulator(n, f)
    result = pbft.run_consensus_round("Block #1")
    safe = f <= pbft.max_faults
    print(f"{n:>7} {f:>8} {pbft.max_faults:>7} {'Yes' if safe else 'No':>7} "
          f"{'Yes' if result['consensus'] else 'No':>11} {result['messages']:>10,}")

print(f"\nMessage complexity: O(n^2) -- scales poorly with validator count.")

In [ ]:
# Visualize PBFT message complexity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Message count vs node count
node_counts = np.arange(4, 201)
msg_counts = []
for n in node_counts:
    f = (n - 1) // 3
    honest = n - f
    # pre-prepare + prepare + commit
    msgs = (n - 1) + honest * (n - 1) + honest * (n - 1)
    msg_counts.append(msgs)

axes[0].plot(node_counts, msg_counts, 'b-', linewidth=2)
axes[0].set_xlabel('Number of Nodes')
axes[0].set_ylabel('Messages per Round')
axes[0].set_title('PBFT Message Complexity: O(n²)')

# Annotate key points
for n_mark in [4, 10, 50, 100, 200]:
    idx = n_mark - 4
    if idx < len(msg_counts):
        axes[0].plot(n_mark, msg_counts[idx], 'ro', markersize=6)
        axes[0].annotate(f'n={n_mark}: {msg_counts[idx]:,}',
                        (n_mark, msg_counts[idx]), fontsize=8,
                        xytext=(10, 10), textcoords='offset points')

# Fault tolerance comparison
systems = {
    'PBFT (3f+1)': lambda n: (n - 1) // 3,
    'PoW (51%)': lambda n: n // 2,
    'PoS-BFT (2/3)': lambda n: (n - 1) // 3,
}

for name, func in systems.items():
    tolerances = [func(n) / n * 100 for n in node_counts]
    axes[1].plot(node_counts, tolerances, linewidth=2, label=name)

axes[1].set_xlabel('Number of Nodes')
axes[1].set_ylabel('Max Fault Tolerance (%)')
axes[1].set_title('Fault Tolerance by Consensus Mechanism')
axes[1].legend()
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.savefig('/tmp/pbft_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("PBFT analysis complete.")
print("Quadratic messaging makes PBFT impractical for >100 nodes.")

---
## 5. Finality Analysis

Different consensus mechanisms offer different types of finality:

- **Probabilistic finality (PoW)**: Confidence in a transaction increases with each confirmation. A transaction is never 100% final, but the probability of reversal decreases exponentially.
- **Deterministic finality (BFT)**: Once a block is committed, it cannot be reverted (assuming <1/3 Byzantine nodes).

> **Definition: Finality** -- The guarantee that a confirmed transaction cannot be altered, reversed, or canceled. Probabilistic finality means the probability of reversal approaches zero; deterministic finality means reversal is mathematically impossible.

In [ ]:
def pow_confirmation_probability(attacker_hashrate: float,
                                  confirmations: int) -> float:
    """Calculate probability an attacker can reverse a PoW transaction.
    
    Uses the formula from Nakamoto's whitepaper.
    
    Args:
        attacker_hashrate: Fraction of total hashrate (0-1)
        confirmations: Number of confirmations
    
    Returns:
        Probability of successful double-spend
    """
    q = attacker_hashrate
    p = 1 - q
    
    if q >= p:
        return 1.0  # Attacker has majority
    
    # Nakamoto's formula (Poisson approximation)
    lam = confirmations * (q / p)
    total = 0.0
    for k in range(confirmations + 1):
        poisson = np.exp(-lam) * lam**k / np.math.factorial(k)
        prob = 1 - (q / p)**(confirmations - k) if confirmations > k else 1
        total += poisson * (1 - prob)
    
    return min(1.0, total)


print("=" * 60)
print("FINALITY ANALYSIS")
print("=" * 60)

# PoW: Double-spend probability vs confirmations
print(f"\nPoW Double-Spend Probability (Nakamoto formula):")
print(f"\n{'Confirmations':>14}", end='')
attacker_rates = [0.10, 0.20, 0.30, 0.40, 0.45]
for rate in attacker_rates:
    print(f" {'q='+f'{rate:.0%}':>10}", end='')
print()
print("-" * 66)

for conf in [1, 2, 3, 6, 12, 24, 60]:
    print(f"{conf:>14}", end='')
    for rate in attacker_rates:
        prob = pow_confirmation_probability(rate, conf)
        if prob < 0.0001:
            print(f" {'<0.01%':>10}", end='')
        else:
            print(f" {prob*100:>9.4f}%", end='')
    print()

print(f"\nBitcoin: 6 confirmations (~1 hour) is standard for most transactions.")
print(f"At q=10%, 6 confirms gives <0.1% attack probability.")

In [ ]:
# Visualize finality comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PoW confirmation probabilities
confirmations = np.arange(1, 31)
for rate in [0.10, 0.20, 0.30, 0.40]:
    probs = [pow_confirmation_probability(rate, c) for c in confirmations]
    axes[0].semilogy(confirmations, probs, linewidth=2, label=f'q={rate:.0%}')

axes[0].axhline(y=0.001, color='red', linestyle='--', alpha=0.5, label='0.1% threshold')
axes[0].set_xlabel('Confirmations')
axes[0].set_ylabel('Attack Success Probability')
axes[0].set_title('PoW: Probabilistic Finality')
axes[0].legend(fontsize=8)
axes[0].set_ylim(1e-8, 1)

# Right: Time to finality comparison
mechanisms = {
    'Bitcoin (PoW)': 60,          # 6 confirmations * 10 min
    'Ethereum (PoS)': 12.8,       # 2 epochs * 6.4 min
    'Solana (PoH+PoS)': 0.4/60,   # ~400ms
    'Cosmos (Tendermint)': 6/60,   # ~6 seconds
    'Avalanche': 2/60,             # ~2 seconds
    'PBFT (permissioned)': 1/60,   # ~1 second
}

names = list(mechanisms.keys())
times = list(mechanisms.values())
colors = ['orange', 'blue', 'purple', 'green', 'red', 'gray']

bars = axes[1].barh(names, times, color=colors)
axes[1].set_xlabel('Time to Finality (minutes)')
axes[1].set_title('Time to Finality by Mechanism')
axes[1].set_xscale('log')

for bar, t in zip(bars, times):
    if t >= 1:
        label = f'{t:.0f} min'
    else:
        label = f'{t*60:.0f}s'
    axes[1].text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2,
                label, va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/finality_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Finality comparison: BFT-based systems achieve seconds, PoW takes minutes to hours.")

---
## 6. Nakamoto Consensus & 51% Attacks

Bitcoin's Nakamoto Consensus uses the **longest chain rule**: the valid chain with the most cumulative proof of work is the canonical chain.

A **51% attack** occurs when an attacker controls more than half the hashrate, allowing them to:
- Double-spend transactions
- Censor specific transactions
- Rewrite recent blockchain history

The cost of a 51% attack depends on:
- Total network hashrate
- Hardware and electricity costs
- Number of confirmations to reverse

In [ ]:
def simulate_51_attack(attacker_pct: float, target_confirmations: int,
                        n_simulations: int = 10000) -> Dict:
    """Monte Carlo simulation of a 51% attack.
    
    Args:
        attacker_pct: Attacker's hashrate as fraction (e.g., 0.51)
        target_confirmations: Confirmations to reverse
        n_simulations: Number of Monte Carlo trials
    """
    successes = 0
    blocks_needed_list = []
    
    for _ in range(n_simulations):
        # Honest chain has `target_confirmations` blocks head start
        honest_lead = target_confirmations
        attacker_blocks = 0
        max_rounds = target_confirmations * 20  # Safety limit
        
        for round_num in range(max_rounds):
            # Each round: either attacker or honest miner finds block
            if np.random.random() < attacker_pct:
                attacker_blocks += 1
            else:
                honest_lead += 1
            
            if attacker_blocks >= honest_lead:
                successes += 1
                blocks_needed_list.append(round_num + 1)
                break
    
    return {
        'success_rate': successes / n_simulations,
        'avg_blocks_needed': np.mean(blocks_needed_list) if blocks_needed_list else float('inf'),
        'median_blocks_needed': np.median(blocks_needed_list) if blocks_needed_list else float('inf'),
        'attacker_pct': attacker_pct,
        'confirmations': target_confirmations
    }


# Simulate attacks at different hashrate levels
np.random.seed(42)

print("=" * 60)
print("51% ATTACK SIMULATION (Monte Carlo)")
print("=" * 60)

print(f"\n{'Hashrate':>10} {'Confirms':>10} {'Success Rate':>14} {'Avg Blocks':>12}")
print("-" * 50)

for pct in [0.30, 0.40, 0.45, 0.49, 0.51, 0.55, 0.60]:
    for conf in [1, 6]:
        result = simulate_51_attack(pct, conf, 5000)
        print(f"{pct:>9.0%} {conf:>10} {result['success_rate']:>13.2%} "
              f"{result['avg_blocks_needed']:>11.0f}")

print(f"\nWith >50% hashrate, attack ALWAYS succeeds (given enough time).")
print(f"With <50%, success drops exponentially with more confirmations.")

---
## 7. Consensus Mechanism Comparison

Each consensus mechanism makes different tradeoffs across key dimensions:

| Dimension | PoW | PoS | PBFT |
|-----------|-----|-----|------|
| Throughput (TPS) | Low (~7) | Medium (~30-100) | High (~1000+) |
| Finality | Probabilistic | Economic | Deterministic |
| Energy | Very High | Very Low | Very Low |
| Decentralization | High | Medium-High | Low |
| Fault Tolerance | 50% hashrate | 33% stake | 33% nodes |

In [ ]:
# Comprehensive comparison
@dataclass
class ConsensusProfile:
    """Performance profile of a consensus mechanism."""
    name: str
    tps: float
    finality_seconds: float
    energy_kwh_per_tx: float
    max_validators: int
    fault_tolerance_pct: float
    decentralization_score: float  # 0-10


profiles = [
    ConsensusProfile("Bitcoin (PoW)", 7, 3600, 700, 1000000, 50, 9),
    ConsensusProfile("Ethereum (PoS)", 30, 768, 0.003, 900000, 33, 8),
    ConsensusProfile("Solana (PoH+PoS)", 4000, 0.4, 0.0005, 2000, 33, 5),
    ConsensusProfile("Cosmos (Tendermint)", 1000, 6, 0.001, 175, 33, 6),
    ConsensusProfile("Hyperledger (PBFT)", 3000, 1, 0.001, 20, 33, 2),
]

print("=" * 90)
print("CONSENSUS MECHANISM COMPARISON")
print("=" * 90)

print(f"\n{'Mechanism':<22} {'TPS':>6} {'Finality':>10} {'Energy/TX':>12} {'Validators':>12} {'Decentralization':>18}")
print("-" * 84)
for p in profiles:
    if p.finality_seconds >= 60:
        fin = f"{p.finality_seconds/60:.0f} min"
    else:
        fin = f"{p.finality_seconds:.1f}s"
    print(f"{p.name:<22} {p.tps:>6,.0f} {fin:>10} {p.energy_kwh_per_tx:>10.3f} kWh "
          f"{p.max_validators:>11,} {p.decentralization_score:>14}/10")

# Radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = ['Throughput', 'Finality\n(fast)', 'Energy\nEfficiency',
              'Scalability', 'Fault\nTolerance', 'Decentralization']
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

colors = ['orange', 'blue', 'purple', 'green', 'gray']

for p, color in zip(profiles, colors):
    # Normalize each metric to 0-10
    values = [
        min(10, np.log10(p.tps + 1) * 3),           # Throughput (log scale)
        min(10, 10 - np.log10(p.finality_seconds + 0.1) * 2.5),  # Finality (inverse)
        min(10, 10 - np.log10(p.energy_kwh_per_tx + 0.0001) * 2),  # Energy (inverse)
        min(10, np.log10(p.max_validators + 1) * 2),  # Scalability
        p.fault_tolerance_pct / 5,                     # Fault tolerance
        p.decentralization_score,                      # Decentralization
    ]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=p.name, alpha=0.7)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 10)
ax.set_title('Consensus Mechanism Comparison', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/consensus_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("No mechanism dominates all dimensions -- each makes tradeoffs.")

---
## Exercises

### Exercise 1: Delegated Proof of Stake (DPoS)

Implement a DPoS simulator where token holders vote for delegates who produce blocks in round-robin order.

**Hints:**
- Token holders vote for delegates proportional to their stake
- Top N delegates (e.g., 21 for EOS-like) are selected as block producers
- Block producers take turns in round-robin
- Analyze centralization vs throughput tradeoffs

In [ ]:
class DPoSSimulator:
    """Delegated Proof of Stake simulator."""
    
    def __init__(self, voter_stakes: Dict[str, float],
                 n_delegates: int = 21) -> None:
        """Initialize DPoS."""
        self.voter_stakes = voter_stakes
        self.n_delegates = n_delegates
        # YOUR CODE HERE
    
    def vote_for_delegates(self, votes: Dict[str, List[str]]) -> List[str]:
        """Token holders vote for delegate candidates."""
        # YOUR CODE HERE
        pass
    
    def produce_blocks(self, n_rounds: int) -> List[str]:
        """Produce blocks in round-robin order."""
        # YOUR CODE HERE
        pass

### Exercise 2: Fork Choice Rule Simulator

Build a simulator that demonstrates different fork choice rules (longest chain, heaviest chain, GHOST protocol).

**Hints:**
- Create a tree structure for the blockchain (with forks)
- Implement longest chain (Bitcoin), heaviest subtree (GHOST), and latest message (Casper)
- Show how each rule selects the canonical chain differently

In [ ]:
class ForkChoiceSimulator:
    """Simulator for different fork choice rules."""
    
    def __init__(self) -> None:
        """Initialize with genesis block."""
        self.blocks: Dict[str, Dict] = {}  # hash -> block
        # YOUR CODE HERE
    
    def add_block(self, parent_hash: str, miner: str,
                  weight: float = 1.0) -> str:
        """Add a block to the tree."""
        # YOUR CODE HERE
        pass
    
    def longest_chain(self) -> List[str]:
        """Select canonical chain by longest chain rule."""
        # YOUR CODE HERE
        pass
    
    def ghost(self) -> List[str]:
        """Select canonical chain by GHOST protocol."""
        # YOUR CODE HERE
        pass

### Exercise 3: Consensus Under Network Partitions

Simulate how different consensus mechanisms behave during network partitions (when the network splits into two groups that can't communicate).

**Hints:**
- Split validators into two partitions
- PoW: both partitions continue mining, creating forks
- BFT: neither partition can reach quorum (safety preserved, liveness lost)
- Analyze what happens when the partition heals

In [ ]:
class NetworkPartitionSimulator:
    """Simulate consensus behavior under network partitions."""
    
    def __init__(self, mechanism: str, n_nodes: int) -> None:
        """Initialize simulator."""
        self.mechanism = mechanism
        self.n_nodes = n_nodes
        # YOUR CODE HERE
    
    def partition(self, split_ratio: float = 0.5) -> None:
        """Split the network into two partitions."""
        # YOUR CODE HERE
        pass
    
    def simulate_partition(self, duration_blocks: int) -> Dict:
        """Simulate consensus during partition."""
        # YOUR CODE HERE
        pass
    
    def heal_partition(self) -> Dict:
        """Simulate partition healing and chain reconciliation."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] The Byzantine Generals Problem and BFT requirements (n >= 3f+1)
- [x] Proof of Work mining simulation and exponential block time distribution
- [x] Proof of Stake validator selection, rewards, and slashing mechanics
- [x] PBFT three-phase commit protocol and O(n^2) message complexity
- [x] Probabilistic vs deterministic finality and confirmation analysis
- [x] 51% attack success probabilities via Monte Carlo simulation
- [x] Consensus mechanism comparison across performance dimensions

### Key Takeaways
1. **No perfect consensus exists** -- the FLP theorem proves deterministic consensus is impossible in fully asynchronous systems
2. **PoW provides probabilistic finality** -- security increases exponentially with confirmations
3. **PoS eliminates energy waste** but introduces new challenges (nothing-at-stake, long-range attacks)
4. **BFT achieves instant finality** but doesn't scale beyond ~100 validators due to O(n^2) messages
5. **The blockchain trilemma is real** -- no mechanism excels at all of security, scalability, and decentralization
6. **51% attacks are about economics** -- the cost of attack must exceed the potential profit

### Further Reading
- Lamport, L. et al. (1982). "The Byzantine Generals Problem." ACM TOPLAS.
- Castro, M. & Liskov, B. (1999). "Practical Byzantine Fault Tolerance." OSDI.
- Buterin, V. (2020). "Why Proof of Stake." vitalik.eth.limo

### Next Steps
- [Notebook 15: Multichain Analysis](15-multichain-analysis.ipynb) -- Cross-chain architecture and interoperability
- [Section 02: Bitcoin Deep Dive](../sections/02-bitcoin-deep-dive.md) -- Bitcoin consensus in depth
- [Section 05: Platform Comparison](../sections/05-platform-comparison.md) -- Architecture tradeoffs